# Gradient boosting: the intuition
Gradient boosting for regression, by hand on five students and then on 100 points of a noisy curve.
The playground is `app.py` (`python app.py`, then open http://127.0.0.1:8050); the cells below reuse its code.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.tree import DecisionTreeRegressor, export_text

## 1. Five students: IQ, CGPA and package (LPA)

In [ ]:
df = pd.DataFrame({"IQ": [90, 100, 110, 120, 80], "CGPA": [8, 7, 6, 9, 5], "LPA": [3, 4, 8, 6, 3]})
X_toy, y_toy = df[["IQ", "CGPA"]].values, df["LPA"].values.astype(float)

# stage 1: the first model is just the mean of the target
df["pred1"] = y_toy.mean()
# pseudo-residuals of stage 1: actual minus predicted
df["res1"] = df["LPA"] - df["pred1"]
df

## 2. Tree 1 learns the residuals, not the package

In [ ]:
# input: IQ and CGPA; target: the residuals of stage 1
tree1 = DecisionTreeRegressor(random_state=0).fit(X_toy, df["res1"])
print(export_text(tree1, feature_names=["IQ", "CGPA"]))
# on the training rows the full-depth tree reproduces the residuals exactly
print(tree1.predict(X_toy))

In [ ]:
# without a learning rate: mean + tree 1 gives back every package exactly (overfitting)
print(4.8 + tree1.predict(X_toy))
# with learning rate 0.1: a small step in the right direction
lr = 0.1
df["pred2"] = df["pred1"] + lr * tree1.predict(X_toy)
df["res2"] = df["LPA"] - df["pred2"]
df

## 3. Tree 2 learns the new residuals

In [ ]:
tree2 = DecisionTreeRegressor(random_state=0).fit(X_toy, df["res2"])
df["pred3"] = df["pred2"] + lr * tree2.predict(X_toy)
df["res3"] = df["LPA"] - df["pred3"]
df.round(3)

In [ ]:
# a new student: IQ 60, CGPA 4.9
new = np.array([[60, 4.9]])
print(4.8 + lr * tree1.predict(new) + lr * tree2.predict(new))

## 4. A noisy curve: 100 points on y = 3x² plus noise

In [ ]:
from app import GRID, X, X_test, boost, figure, mse, predict, y, y_test

print("mean of y (stage 1):", y.mean().round(6))
# three trees with 8 leaves each, learning rate 1, as in the step-by-step walk-through
for m in range(4):
    f0, trees = boost(m, 1.0, 8)
    print(m, "trees: training MSE", round(mse(y, predict(f0, trees, 1.0, X)), 4))

In [ ]:
# the ensemble after 3 trees (interactive Plotly chart)
figure(n_trees=3, learning_rate=1.0)

In [ ]:
# 50 trees with learning rate 1: the curve chases every point
figure(n_trees=50, learning_rate=1.0)

## 5. Our loop equals scikit-learn's GradientBoostingRegressor

In [ ]:
# GradientBoostingRegressor also starts from the mean and fits squared-error trees to the residuals.
# max_depth=None so that only max_leaf_nodes limits the trees, as in our loop.
for lr in (1.0, 0.1):
    f0, trees = boost(50, lr, 8)
    gbr = GradientBoostingRegressor(loss="squared_error", n_estimators=50, learning_rate=lr,
                                    max_leaf_nodes=8, max_depth=None, random_state=42).fit(X, y)
    gap = max(np.abs(p - predict(f0, trees[:m], lr, X)).max()
              for m, p in enumerate(gbr.staged_predict(X), start=1))
    print(f"learning rate {lr}: largest difference over 50 stages = {gap:.1e}")

## 6. Learning rate and number of trees

In [ ]:
# 100 test points give a noisy test error, so average 20 fresh datasets made with the same recipe,
# each scored on 5,000 new points. GradientBoostingRegressor matches our loop (checked above).
res = {lr: [] for lr in (1.0, 0.5, 0.1)}
for s in range(20):
    r = np.random.RandomState(s)
    Xs = r.rand(100, 1) - 0.5
    ys = 3 * Xs[:, 0] ** 2 + 0.05 * r.randn(100)
    Xt = r.rand(5000, 1) - 0.5
    yt = 3 * Xt[:, 0] ** 2 + 0.05 * r.randn(5000)
    for lr in res:
        gbr = GradientBoostingRegressor(n_estimators=200, learning_rate=lr, max_leaf_nodes=8, max_depth=None,
                                        random_state=0).fit(Xs, ys)
        res[lr].append([mse(yt, p) for p in gbr.staged_predict(Xt)])
for lr, runs in res.items():
    test = np.mean(runs, axis=0)
    print(f"learning rate {lr}: best test MSE {test.min():.5f} after {test.argmin() + 1} trees,"
          f" {test[2]:.5f} after 3, {test[49]:.5f} after 50, {test[-1]:.5f} after 200")

## 7. Playground
Change the number of trees, the learning rate and the leaves per tree. Run `python app.py` in this folder and open
http://127.0.0.1:8050, or set `RUN_APP = True` to show it here.

In [ ]:
RUN_APP = False
if RUN_APP:
    from app import app
    app.run(jupyter_mode="inline")